In [1]:
!pip install fastf1 lightgbm torch pandas numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
import numpy as np
import fastf1

# Create cache directory if it doesn't exist
cache_dir = 'f1_cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# Enable local caching
fastf1.Cache.enable_cache(cache_dir) 

def load_race_data(years):
    records = []
    
    for year in years:
        schedule = fastf1.get_event_schedule(year)
        races = schedule[schedule['EventFormat'] == 'conventional']
        
        for _, event in races.iterrows():
            round_num = event['RoundNumber']
            try:
                session = fastf1.get_session(year, round_num, 'R')
                # Load only results and weather (telemetry=False speeds up processing)
                session.load(telemetry=False, laps=False, weather=True)
                
                results = session.results
                weather = session.weather_data
                
                avg_track_temp = weather['TrackTemp'].mean() if not weather.empty else 30.0
                
                for _, driver in results.iterrows():
                    records.append({
                        'season': year,
                        'round': round_num,
                        'driver': driver['Abbreviation'],
                        'constructor': driver['TeamName'],
                        'grid': driver['GridPosition'],
                        'track_temp': avg_track_temp,
                        'position': driver['Position']
                    })
            except Exception as e:
                continue
                
    df = pd.DataFrame(records)
    df = df.dropna(subset=['position', 'grid'])
    df['grid'] = df['grid'].astype(int)
    df['position'] = df['position'].astype(int)
    return df

# Download race data for recent seasons
df_raw = load_race_data([2023, 2024])
df_raw.head()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading d

,season,round,driver,constructor,grid,track_temp,position
0,2023,1,VER,Red Bull Racing,1,31.011801,1
1,2023,1,PER,Red Bull Racing,2,31.011801,2
2,2023,1,ALO,Aston Martin,5,31.011801,3
3,2023,1,SAI,Ferrari,4,31.011801,4
4,2023,1,HAM,Mercedes,7,31.011801,5


In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = df_raw.copy()

# Convert driver code & team name strings into numbers
le_driver = LabelEncoder()
le_team = LabelEncoder()

df['driver_code'] = le_driver.fit_transform(df['driver'])
df['team_code'] = le_team.fit_transform(df['constructor'])

# Define model input features and the target prediction column
features = ['grid', 'driver_code', 'team_code', 'track_temp']
target = 'position'

# Time-based Train/Test split: Train on 2023 season, test on 2024 season
train_mask = df['season'] < 2024
test_mask = df['season'] >= 2024

X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, target]
X_test = df.loc[test_mask, features]
y_test = df.loc[test_mask, target]

# Scale numeric features for Neural Network training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training dataset size: {len(X_train)} rows")
print(f"Testing dataset size: {len(X_test)} rows")

Training dataset size: 319 rows
Testing dataset size: 359 rows


In [4]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Initialize and train LightGBM Regressor
lgb_model = lgb.LGBMRegressor(
    n_estimators=150, 
    learning_rate=0.03, 
    max_depth=4, 
    random_state=42
)

lgb_model.fit(X_train, y_train)

# Generate predictions on 2024 test data
lgb_preds = lgb_model.predict(X_test)

print("--- LightGBM Model Performance ---")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, lgb_preds):.2f} positions off")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, lgb_preds)):.2f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000257 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 70
[LightGBM] [Info] Number of data points in the train set: 319, number of used features: 4
[LightGBM] [Info] Start training from score 10.470219
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define Multi-Layer Perceptron Network Architecture
class F1Predictor(nn.Module):
    def __init__(self, input_dim):
        super(F1Predictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.network(x)

# Convert scaled NumPy arrays into PyTorch Tensors
X_t_train = torch.tensor(X_train_scaled, dtype=torch.float32)
y_t_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_t_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_t_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Initialize Model, Loss Function, and Optimizer
nn_model = F1Predictor(input_dim=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(nn_model.parameters(), lr=0.005)

# Neural Network Training Loop
epochs = 200
nn_model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = nn_model(X_t_train)
    loss = criterion(outputs, y_t_train)
    loss.backward()
    optimizer.step()

# Evaluation on Test Data
nn_model.eval()
with torch.no_grad():
    nn_preds = nn_model(X_t_test).numpy().flatten()

print("--- PyTorch Neural Network Performance ---")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, nn_preds):.2f} positions off")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, nn_preds)):.2f}")

--- PyTorch Neural Network Performance ---
Mean Absolute Error: 3.18 positions off
Root Mean Squared Error: 4.17


In [7]:
# 1. Compare actual vs predicted for a test race
sample_test = df[test_mask].copy()
sample_test['lgb_pred_pos'] = lgb_preds
sample_test['nn_pred_pos'] = nn_preds

# Filter for the first race of the test season and sort by actual finish
single_race = sample_test[(sample_test['round'] == 1)].sort_values(by='position')

comparison = single_race[['driver', 'grid', 'position', 'lgb_pred_pos', 'nn_pred_pos']].copy()
comparison.columns = ['Driver', 'Grid Start', 'Actual Finish', 'LGBM Predicted', 'NN Predicted']

print("=== Model Comparison: Test Race 1 ===")
print(comparison.head(10).to_string(index=False))

# 2. Map upcoming race data using exact team names present in training data
# Safe lookup helper to prevent KeyError/ValueError for unseen labels
def safe_encode(encoder, values):
    known_classes = set(encoder.classes_)
    default_class = encoder.classes_[0] # fallback to first known class if unknown
    cleaned_values = [v if v in known_classes else default_class for v in values]
    return encoder.transform(cleaned_values)

# Get valid team/driver names directly from your encoder classes
print("\nSample Known Teams:", le_team.classes_[:5])

upcoming_race_data = pd.DataFrame([
    {'grid': 1, 'driver': df['driver'].iloc[0], 'constructor': df['constructor'].iloc[0], 'track_temp': 38.0},
    {'grid': 2, 'driver': df['driver'].iloc[1], 'constructor': df['constructor'].iloc[1], 'track_temp': 38.0},
    {'grid': 3, 'driver': df['driver'].iloc[2], 'constructor': df['constructor'].iloc[2], 'track_temp': 38.0},
    {'grid': 4, 'driver': df['driver'].iloc[3], 'constructor': df['constructor'].iloc[3], 'track_temp': 38.0},
])

# Process features safely
upcoming_race_data['driver_code'] = safe_encode(le_driver, upcoming_race_data['driver'])
upcoming_race_data['team_code'] = safe_encode(le_team, upcoming_race_data['constructor'])

X_upcoming = upcoming_race_data[features]
X_upcoming_scaled = scaler.transform(X_upcoming)

# LightGBM Prediction
upcoming_race_data['LGBM_Pred'] = lgb_model.predict(X_upcoming)

# PyTorch NN Prediction
nn_model.eval()
with torch.no_grad():
    X_upcoming_tensor = torch.tensor(X_upcoming_scaled, dtype=torch.float32)
    upcoming_race_data['NN_Pred'] = nn_model(X_upcoming_tensor).numpy().flatten()

print("\n=== Upcoming Race Predictions ===")
print(upcoming_race_data[['driver', 'constructor', 'grid', 'LGBM_Pred', 'NN_Pred']].to_string(index=False))

=== Model Comparison: Test Race 1 ===
Driver  Grid Start  Actual Finish  LGBM Predicted  NN Predicted
   VER           1              1        2.597904      6.657381
   PER           5              2        7.722161      7.043638
   SAI           4              3        6.569351      9.413626
   LEC           2              4        6.136966      8.548691
   RUS           3              5        9.690015      7.500088
   NOR           7              6       11.806889      8.338743
   HAM           9              7        5.645885      9.327228
   PIA           8              8       11.791000      8.615023
   ALO           6              9        6.311287      9.408268
   STR          12             10       10.262877     11.169350

Sample Known Teams: ['Alfa Romeo' 'AlphaTauri' 'Alpine' 'Aston Martin' 'Ferrari']

=== Upcoming Race Predictions ===
driver     constructor  grid  LGBM_Pred  NN_Pred
   VER Red Bull Racing     1   2.157202 3.447920
   PER Red Bull Racing     2   3.593231 3.

In [21]:
cd C:\Users\admin

C:\Users\admin


In [22]:
rmdir /s /q .git

In [24]:
!git init
!git branch -M main
!git remote add origin https://github.com/Nandhini-2006/Formula1-LapToLap-ML-Pipeline.git
!git add FORMULA_1.ipynb
!git commit -m "Add Formula 1 race outcome predictor notebook"
!git push -u origin main --force

Initialized empty Git repository in C:/Users/admin/.git/


The file will have its original line endings in your working directory


[main (root-commit) 4f57d39] Add Formula 1 race outcome predictor notebook
 1 file changed, 1093 insertions(+)
 create mode 100644 FORMULA_1.ipynb
branch 'main' set up to track 'origin/main'.


To https://github.com/Nandhini-2006/Formula1-LapToLap-ML-Pipeline.git
 * [new branch]      main -> main
